# 09 — Interpretability

TreeSHAP global and local explanations, partial dependence plots, and permutation feature importance for the calibrated XGBoost model.

In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

MODELS_DIR  = os.path.abspath('../models')
FIGURES_DIR = os.path.abspath('../figures')
DATA_DIR    = os.path.abspath('../data/processed')
print("Paths OK")


In [ ]:
# ── Load model and data ───────────────────────────────────────
def load_all():
    cal_path = os.path.join(MODELS_DIR, 'xgb_calibrated.joblib')
    parquet  = os.path.join(DATA_DIR, 'modeling_table.parquet')

    if os.path.exists(parquet):
        df = pd.read_parquet(parquet)
        feature_cols = [c for c in df.columns
                        if c not in ['is_delayed', 'route_id', 'station_id', 'timestamp']]
        X = df[feature_cols].values.astype(float)
        y = df['is_delayed'].values
        from sklearn.model_selection import train_test_split
        X_tv, X_test, y_tv, y_test = train_test_split(
            X, y, test_size=0.15, random_state=RANDOM_STATE, stratify=y)
        X_train, X_val, y_train, y_val = train_test_split(
            X_tv, y_tv, test_size=0.176, random_state=RANDOM_STATE, stratify=y_tv)
        X_train_df = pd.DataFrame(X_train, columns=feature_cols)
        X_test_df  = pd.DataFrame(X_test,  columns=feature_cols)
    else:
        from sklearn.datasets import make_classification
        from sklearn.model_selection import train_test_split
        X_all, y_all = make_classification(n_samples=12000, n_features=20,
                                           n_informative=12, weights=[0.65, 0.35],
                                           random_state=RANDOM_STATE)
        feature_cols = [f'feat_{i}' for i in range(20)]
        X_tv, X_test, y_tv, y_test = train_test_split(
            X_all, y_all, test_size=0.15, random_state=RANDOM_STATE, stratify=y_all)
        X_train, X_val, y_train, y_val = train_test_split(
            X_tv, y_tv, test_size=0.176, random_state=RANDOM_STATE, stratify=y_tv)
        X_train_df = pd.DataFrame(X_train, columns=feature_cols)
        X_test_df  = pd.DataFrame(X_test,  columns=feature_cols)
        feature_cols = feature_cols

    if os.path.exists(cal_path):
        model = joblib.load(cal_path)
    else:
        import xgboost as xgb
        from sklearn.calibration import CalibratedClassifierCV
        base = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05,
                                  use_label_encoder=False, eval_metric='logloss',
                                  random_state=RANDOM_STATE)
        base.fit(X_train, y_train)
        model = CalibratedClassifierCV(base, cv='prefit', method='sigmoid')
        model.fit(X_val, y_val)

    return model, X_train_df, X_val, X_test_df, y_train, y_val, y_test, feature_cols

model, X_train_df, X_val, X_test_df, y_train, y_val, y_test, FEATURE_COLS = load_all()
print(f"Model: {type(model).__name__}")
print(f"Features: {len(FEATURE_COLS)}, Test rows: {len(y_test)}")


In [ ]:
# ── TreeSHAP: subsample training set to 5,000 rows ───────────
import shap

shap.initjs()

# Extract the base XGBoost estimator from the calibrated wrapper
base_estimator = getattr(model, 'estimator', None)
if base_estimator is None:
    # CalibratedClassifierCV stores calibrated_classifiers_
    try:
        base_estimator = model.calibrated_classifiers_[0].base_estimator
    except AttributeError:
        base_estimator = model.calibrated_classifiers_[0].estimator

np.random.seed(RANDOM_STATE)
idx_sub = np.random.choice(len(X_train_df), min(5000, len(X_train_df)), replace=False)
X_sub   = X_train_df.iloc[idx_sub]

explainer    = shap.TreeExplainer(base_estimator)
shap_values  = explainer.shap_values(X_sub)

# For binary XGBoost, shap_values shape is (n, features) for positive class
if isinstance(shap_values, list):
    sv = shap_values[1]
else:
    sv = shap_values

print(f"SHAP values shape: {sv.shape}")


In [ ]:
# ── Global summary bar plot (top 10 features) ─────────────────
fig, ax = plt.subplots(figsize=(9, 6))
mean_abs_shap = np.abs(sv).mean(axis=0)
top10_idx     = np.argsort(mean_abs_shap)[-10:][::-1]
top10_names   = [FEATURE_COLS[i] for i in top10_idx]
top10_vals    = mean_abs_shap[top10_idx]

ax.barh(top10_names[::-1], top10_vals[::-1], color='steelblue', alpha=0.85)
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('Global Feature Importance — TreeSHAP (Top 10)')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'shap_summary.png'), dpi=150)
plt.show()

TOP_FEATURES = top10_names
print("Top 10 features by SHAP:", TOP_FEATURES)


In [ ]:
# ── SHAP waterfall plots: 1 TP, 1 FN, 1 FP ───────────────────
proba_test = model.predict_proba(X_test_df.values)[:, 1]
preds_test = (proba_test >= 0.5).astype(int)

# Compute SHAP on test set
shap_test = explainer.shap_values(X_test_df)
if isinstance(shap_test, list):
    sv_test = shap_test[1]
else:
    sv_test = shap_test

tp_idx = np.where((preds_test == 1) & (y_test == 1))[0]
fn_idx = np.where((preds_test == 0) & (y_test == 1))[0]
fp_idx = np.where((preds_test == 1) & (y_test == 0))[0]

def plot_local_shap(ax, row_idx, label, color):
    shap_row = sv_test[row_idx]
    sorted_i = np.argsort(np.abs(shap_row))[-8:]
    names = [FEATURE_COLS[i] for i in sorted_i]
    vals  = shap_row[sorted_i]
    colors = ['#EF5350' if v > 0 else '#42A5F5' for v in vals]
    ax.barh(names, vals, color=colors, alpha=0.85)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'{label}  (row {row_idx})', fontsize=10, color=color)
    ax.set_xlabel('SHAP value')

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle('Local SHAP Explanations', fontsize=13)
for ax, idx_arr, label, color in [
    (axes[0], tp_idx, 'True Positive',  'green'),
    (axes[1], fn_idx, 'False Negative', 'orange'),
    (axes[2], fp_idx, 'False Positive', 'crimson'),
]:
    if len(idx_arr) > 0:
        plot_local_shap(ax, idx_arr[0], label, color)
    else:
        ax.text(0.5, 0.5, f'No {label}', ha='center', va='center')
        ax.set_title(label)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'shap_local.png'), dpi=150)
plt.show()


In [ ]:
# ── Partial Dependence Plots: top 6 features ─────────────────
from sklearn.inspection import PartialDependenceDisplay

top6_idx = [FEATURE_COLS.index(f) for f in TOP_FEATURES[:6] if f in FEATURE_COLS]
top6_idx = top6_idx[:6]

try:
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    fig.suptitle('Partial Dependence Plots — Top 6 SHAP Features', fontsize=13)

    PartialDependenceDisplay.from_estimator(
        base_estimator, X_test_df, top6_idx,
        feature_names=FEATURE_COLS,
        ax=axes.ravel()[:len(top6_idx)],
        grid_resolution=50,
        kind='average',
    )
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'pdp_grid.png'), dpi=150)
    plt.show()
except Exception as e:
    print(f"PDP failed: {e}. Generating manual PDPs.")
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    fig.suptitle('Partial Dependence Plots — Top 6 SHAP Features', fontsize=13)
    for ax, fidx in zip(axes.ravel(), top6_idx):
        vals = np.linspace(X_test_df.iloc[:, fidx].min(),
                           X_test_df.iloc[:, fidx].max(), 50)
        pdp  = []
        X_tmp = X_test_df.values.copy()
        for v in vals:
            X_tmp[:, fidx] = v
            pdp.append(model.predict_proba(X_tmp)[:, 1].mean())
        ax.plot(vals, pdp, lw=2)
        ax.set_xlabel(FEATURE_COLS[fidx])
        ax.set_ylabel('P(delayed)')
        ax.set_title(FEATURE_COLS[fidx])
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'pdp_grid.png'), dpi=150)
    plt.show()


In [ ]:
# ── Two-feature PDP: top-2 features ──────────────────────────
f1_idx = top6_idx[0]
f2_idx = top6_idx[1]
vals1  = np.linspace(X_test_df.iloc[:, f1_idx].quantile(0.05),
                     X_test_df.iloc[:, f1_idx].quantile(0.95), 20)
vals2  = np.linspace(X_test_df.iloc[:, f2_idx].quantile(0.05),
                     X_test_df.iloc[:, f2_idx].quantile(0.95), 20)
Z = np.zeros((len(vals2), len(vals1)))
X_tmp = X_test_df.values.copy()
for i, v2 in enumerate(vals2):
    for j, v1 in enumerate(vals1):
        X_tmp[:, f1_idx] = v1
        X_tmp[:, f2_idx] = v2
        Z[i, j] = model.predict_proba(X_tmp)[:, 1].mean()

fig, ax = plt.subplots(figsize=(8, 6))
cf = ax.contourf(vals1, vals2, Z, levels=20, cmap='RdYlGn_r')
plt.colorbar(cf, ax=ax, label='P(delayed)')
ax.set_xlabel(FEATURE_COLS[f1_idx])
ax.set_ylabel(FEATURE_COLS[f2_idx])
ax.set_title(f'2-Feature PDP: {FEATURE_COLS[f1_idx]} × {FEATURE_COLS[f2_idx]}')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'pdp_2feature.png'), dpi=150)
plt.show()


In [ ]:
# ── Permutation feature importance ───────────────────────────
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score

print("Computing permutation importance (may take ~60s)...")
perm = permutation_importance(
    model, X_val, y_val,
    n_repeats=10,
    scoring='roc_auc',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

perm_imp = pd.Series(perm.importances_mean, index=FEATURE_COLS).sort_values(ascending=False)
print("\nTop 10 by permutation importance:")
print(perm_imp.head(10).round(5).to_string())


In [ ]:
# ── Compare SHAP vs permutation importance: Spearman ─────────
shap_rank = pd.Series(mean_abs_shap, index=FEATURE_COLS).rank(ascending=False)
perm_rank = perm_imp.rank(ascending=False)
shared    = shap_rank.index.intersection(perm_rank.index)

rho, pval = spearmanr(shap_rank[shared], perm_rank[shared])
print(f"Spearman ρ (SHAP vs Permutation): {rho:.4f}  (p={pval:.4f})")
assert rho > 0.70, f"Spearman ρ={rho:.3f} below 0.70 threshold"
print("✓ Spearman > 0.70 assertion passed")

# Comparison bar chart
fig, ax = plt.subplots(figsize=(10, 5))
top_n   = 12
top_sh  = pd.Series(mean_abs_shap, index=FEATURE_COLS).nlargest(top_n)
top_pm  = perm_imp.reindex(top_sh.index)

x = np.arange(top_n)
ax.bar(x - 0.2, top_sh / top_sh.max(), width=0.35, label='SHAP (norm.)', color='steelblue', alpha=0.8)
ax.bar(x + 0.2, top_pm / top_pm.max(), width=0.35, label='Permutation (norm.)', color='orange', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(top_sh.index, rotation=45, ha='right')
ax.set_ylabel('Normalised importance')
ax.set_title(f'SHAP vs Permutation Importance  (Spearman ρ={rho:.3f})')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'shap_vs_permutation.png'), dpi=150)
plt.show()
